# 传统的离线蒸馏（off-policy）

- 做法：教师模型生成数据 →→ 存到磁盘 →→ 学生模型加载数据 →→ 训练。
- 问题：学生模型训练几步后，它的分布P_student变了，但训练数据还是旧的P_teacher生成的。这就导致了 分布偏移 (Distribution Shift)，也就是前文提到的“off-policy training”


# on-policy distillation

- 做法： 
  1. 学生模型先跑：让学生模型根据当前参数生成一批数（student_trajectory）。
  2. 教师模型后查：<b>拿着学生生成的这些 token，去问教师模型：“如果当时是你，你在这个位置给出这个 token 的概率是多少？”</b>
  3. 计算损失：基于这个概率计算 KL 散度或交叉熵。
  
- 优势：学生永远是在学习“针对自己当前行为”的教师反馈。数据分布始终与学生当前策略对齐（On-Policy）

两个并行的模型实例：
- student runner：负责generate()。需要支持采样
- teacher evaluator：负责compute_logprobs(input_ids)。只需要前向推理


In [ ]:
for step in range(total_steps):
    # 1.  student generate 一个轨迹（on-policy）
    # input：prompt tokens
    
    student_trajectory = student_model.genearate(
        prompts,
        max_length = 512,
        do_sample = True,
        temperature = 0.7
    )
    
    # 2. teacher evaluates 轨迹
    # 我们需要student实际选择的token的log-probs
    
    with torch.no_grad():
        teacher_outputs = teacher_model(
            student_trajectory, # student实际选择token（轨迹）
            output_logits = True   
        )
        # “拿着学生模型实际生成的轨迹（token序列），去查询教师模型在这些特定位置上的对数概率。”
        teacher_log_probs = compute_log_probs(teacher_outputs.logits,student_trajectory)
    
    # 3. 计算loss
    # student尝试在这些指定的token上与teacher的概率分布相匹配
    student_outputs = student_model(student_trajectory)
    student_log_probs = compute_log_probs(student_outputs.logits,student_trajectory)
    
    # 4. 蒸馏损失（KL散度或者MSE在log-probs）
    # 
    loss = kl_divergence(teacher_log_probs,student_log_probs).mean()
    
    # （可选）增加一个奖励信号，如果结果是对的
    # loss = loss - alpha * reward_signal
    
    # 5. 反向传播和更新
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(0)
        
    

## compute_log_probs ：
- 从巨大的logits矩阵中，根据student_trajectory里的token ID，挑出对应的概率值，并取log

In [ ]:
import torch
import torch.nn.functional as F

def compute_log_probs(logtis,target_token_ids):
    '''
    logits: [batch,seq_len,vocab_size]： teacher的logits
    target_token_ids: [batch,seq_len] ：student_trajectory
    
    返回：[batch,seq_len] 的对数概率
    '''
    # 1. 将logits转换为log_softmax(数值更稳定)
    log_probs = F.log_softmax(logits,dim=-1) # [B,L,V]
    
    # 2. 使用gather操作提取对应token的概率
    
    # unsqueeze(-1) 是为了匹配gather的维度要求
    target_ids_expanded = target_token_ids.unsqueeze(-1) # [B,L,1]
    
    # 在vocab维度上提取
    selected_log_probs = log_probs.gather(dim=2,index = target_ids_expanded).squeeze(-1) # [B,L]
    
    return selected_log_probs
   

In [2]:
import torch
import torch.nn.functional as F

# 1. 模拟模型输出 (Batch=2, 序列长度=3, 词表大小=5)
# 假设这是 log_softmax 之后的结果
logits = torch.tensor([
    [[-2.0, -1.0,  0.0, -3.0, -4.0],  # 第1个词: 预测分布
      [-1.5, -0.5, -2.0,  0.0, -3.0],  # 第2个词
      [ 0.0, -2.0, -1.0, -3.0, -4.0]], # 第3个词
     
    [[-1.0, -2.0, -3.0,  0.0, -4.0],
      [-2.0,  0.0, -1.0, -3.0, -4.0],
      [-3.0, -4.0,  0.0, -1.0, -2.0]]
]) 
# shape: [2, 3, 5]

# 2. 模拟目标句子 (Batch=2, 序列长度=3)
# 这里的数字代表“我想取第几个词的概率”
target_ids = torch.tensor([
    [2, 3, 0],  # 样本1: 想取第2、第3、第0个词
    [3, 1, 2]   # 样本2: 想取第3、第1、第2个词
])
# shape: [2, 3]

# 3. 使用 gather 提取
# 我们想在最后一个维度 (dim=2, 即词表维度) 上抓取
# index 需要和 input 维度对齐，所以要把 target_ids 变成 [2, 3, 1]
index = target_ids.unsqueeze(-1) 
# index shape: [2, 3, 1]
print(index)

# 执行抓取
result = torch.gather(logits, dim=2, index=index)

# 把多余的维度 squeeze 掉，变回 [2, 3]
selected_log_probs = result.squeeze(-1)

print("原始 logits 形状:", logits.shape)
print("目标 IDs 形状:", target_ids.shape)
print("抓取结果形状:", selected_log_probs.shape)
print("抓取结果:\n", selected_log_probs)

# 验证一下手动计算对不对：
# 样本1 第1个位置: logits[0, 0, 2] 应该是 0.0 -> 结果里是 0.0 ✅
# 样本1 第2个位置: logits[0, 1, 3] 应该是 0.0 -> 结果里是 0.0 ✅
# 样本2 第2个位置: logits[1, 1, 1] 应该是 0.0 -> 结果里是 0.0 ✅

tensor([[[2],
         [3],
         [0]],

        [[3],
         [1],
         [2]]])
原始 logits 形状: torch.Size([2, 3, 5])
目标 IDs 形状: torch.Size([2, 3])
抓取结果形状: torch.Size([2, 3])
抓取结果:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
